# Project 08: Viterbi Algorithm Analysis
This notebook demonstrates the implementation of a Hidden Markov Model (HMM). We focus on the Viterbi algorithm to find the most likely sequence of hidden states given an observed sequence.

## Main Algorithm

In [1]:
import math


class HMM:
    """
    Base class for Hidden Markov Model data management.
    Handles initialization and log-space conversion for Numerical Stability.
    """

    def __init__(self, states, initial_probs, transition_probs, emission_probs):
        self.states = states
        # Convert all probabilities to log-scale
        self.log_initial = {s: self.convert_to_log_scale(initial_probs[s]) for s in states}

        # log_transition[previous][current]
        self.log_transition = {from_state: {to_state: self.convert_to_log_scale(transition_probs[from_state][to_state])
                                for to_state in states} for from_state in states}

        # log_emission[state][observation]
        self.log_emission = {s: {obs: self.convert_to_log_scale(prob)
                             for obs, prob in emits.items()}
                         for s, emits in emission_probs.items()}

    def convert_to_log_scale(self, prob):
        """Helper to calculate log(p) and handling p=0 as -infinity."""
        if prob <= 0:
            return -float('inf')
        return math.log(prob)


class Viterbi(HMM):
    """
    Subclass implementing the Viterbi Algorithm for Optimal Path Finding.
    Uses Iterative Tabulation.
    """

    def run(self, observation_sequence):
        """
        Executes the Viterbi algorithm on a given sequence of observations.
        """
        n = len(observation_sequence)
        if n == 0:
            return []

        # Initialize Score and Traceback Matrices: O(N x K) Space
        viterbi_matrix = {s: [0.0] * n for s in self.states}
        traceback_matrix = {s: [None] * n for s in self.states}

        # INITIALIZATION (t = 0)
        initial_observation = observation_sequence[0]
        for s in self.states:
            # Score(s,0) = log_initial(s) + log_emission(s, obs[0])
            current_emission= self.log_emission[s].get(initial_observation, -float('inf'))
            viterbi_matrix[s][0] = self.log_initial[s] + current_emission
            traceback_matrix[s][0] = None  # Boundary for traceback

        # ITERATION (t = 1 to N-1)
        for t in range(1, n):
            current_observation = observation_sequence[t]
            for current_state in self.states:

                # Initialize to negative infinity to find the maximum in log-space
                best_prob = -float('inf')
                best_previous_state = None

                # Find the best transition from the previous column
                for previous_state in self.states:
                    # Adds emission once within the maximization loop
                    path_score = viterbi_matrix[previous_state][t - 1] + self.log_transition[previous_state][current_state] + self.log_emission[current_state][observation_sequence[t]]

                    if path_score > best_prob:
                        best_prob = path_score
                        best_previous_state = previous_state

                # Update matrix with path score
                viterbi_matrix[current_state][t] = best_prob

                # Update Traceback Matrix with the ptr
                traceback_matrix[current_state][t] = best_previous_state

        return self.traceback(viterbi_matrix, traceback_matrix, n)

    def traceback(self, viterbi_matrix, traceback_matrix, n):
        """Reconstructing the Optimal Path from the stored pointers."""
        result_path = []
        print(f"Starting traceback from final index t={n-1}")
        # TERMINATION
        final_state = None
        max_final_score = -float('inf')
        for s in self.states:
            if viterbi_matrix[s][n - 1] > max_final_score:
                max_final_score = viterbi_matrix[s][n - 1]
                final_state = s

        # Walk backward from the end to the start
        if final_state is not None:
            result_path.append(final_state)

            # Move from t = N-1 down to t = 1
            for t in range(n - 1, 0, -1):
                final_state = traceback_matrix[final_state][t]
                result_path.append(final_state)

        # Reverse to get chronological order (Time 0 to Time N-1)
        result_path.reverse()
        return result_path

print("Successfully initialized HMM and Viterbi classes.")

Successfully initialized HMM and Viterbi classes.


## Test 1: CpG Island Detection
This was the example provided as part of the assignment

In [2]:
# Define Parameters
# States are I = CpG Island, G = Genomic Background
dna_states = ['I', 'G']

# Initial Probabilities
dna_initial_probs = {
    "I": 0.2,
    "G": 0.8
}

# Transition Probabilities
dna_transition_probs = {
    "I": {"I": 0.7, "G": 0.3},
    "G": {"I": 0.1, "G": 0.9}
}

# Emission Probabilities 
dna_emission_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}

# Initialize the Model
dna_model = Viterbi(dna_states, dna_initial_probs, dna_transition_probs, dna_emission_probs)

# Define the DNA Observation Sequence
dna_observation = "GGCACTGAA"

# Execute the Viterbi Algorithm
print(f"Input DNA Sequence: {dna_observation}")

dna_path = dna_model.run(list(dna_observation))

# Display Results
print("-" * 50)
print(f"Sequence: {'  '.join(dna_observation)}")
print("-" * 50)
print(f"States:   {'  '.join(dna_path)}")
print("-" * 50)

Input DNA Sequence: GGCACTGAA
Starting traceback from final index t=8
--------------------------------------------------
Sequence: G  G  C  A  C  T  G  A  A
--------------------------------------------------
States:   G  G  G  G  G  G  G  G  G
--------------------------------------------------


## Test 2: High-Contrast Emission Test
In this test case, we evaluate how the algorithm balances immediate evidence against the overall biological context. By setting extreme emission differences (90% GC-content in Islands vs. 10% in Background) and reducing transition penalties, we observe the point where the model decides the biological context has shifted. This proves our implementation can accurately identify sharp boundaries between CpG Islands and genomic background, ensuring that the predicted hidden states are driven by the most significant biological signals

In [3]:
# Define Parameters
contrast_states = ["I", "G"]
contrast_observation = "GGCACTGAA"

# Equal probabilities to allow the observations to drive the first state
contrast_initial_probs = {"I": 0.5, "G": 0.5}

# Lowered transition penalties 
contrast_transition_probs = {
    "I": {"I": 0.6, "G": 0.4},
    "G": {"I": 0.4, "G": 0.6}
}

# Altered emission differences between GC and AT
contrast_emission_probs = {
    "I": {"A": 0.05, "C": 0.45, "G": 0.45, "T": 0.05},
    "G": {"A": 0.45, "C": 0.05, "G": 0.05, "T": 0.45}
}

# Run the Model
contrast_viterbi = Viterbi(contrast_states, contrast_initial_probs, contrast_transition_probs, contrast_emission_probs)
contrast_path = contrast_viterbi.run(list(contrast_observation))

# Print Results
print("-"*50)
print(f"DNA Sequence:  {'  '.join(list(contrast_observation))}")
print("-"*50)
print(f"Optimal Path:  {'  '.join(contrast_path)}")
print("-"*50)

Starting traceback from final index t=8
--------------------------------------------------
DNA Sequence:  G  G  C  A  C  T  G  A  A
--------------------------------------------------
Optimal Path:  I  I  I  G  I  G  I  G  G
--------------------------------------------------


## Test 3: Loaded Dice
Another test for a common case of a loaded and fair die that we wanted to test out.

In [4]:
# Define Model Parameters
states = ['Fair', 'Loaded']
initial_probs = {'Fair': 0.9, 'Loaded': 0.1}

transition_probs = {
    'Fair':   {'Fair': 0.95, 'Loaded': 0.05},
    'Loaded': {'Fair': 0.10, 'Loaded': 0.90}
}

emission_probs = {
    'Fair':   {'1': 1/6, '2': 1/6, '3': 1/6, '4': 1/6, '5': 1/6, '6': 1/6},
    'Loaded': {'1': 0.1, '2': 0.1, '3': 0.1, '4': 0.1, '5': 0.1, '6': 0.5}
}

# Initialize Viterbi Model
model = Viterbi(states, initial_probs, transition_probs, emission_probs)

# Define Test Observation Sequence
observations = "123666666121"

# Execute Algorithm
path = model.run(list(observations))

# Display Results
print("-" * 100)
print(f"Observations: {' '.join(observations)}")
print("-" * 100)
print(f"Optimal Path:  {' '.join(path)}")
print("-" * 100)

Starting traceback from final index t=11
----------------------------------------------------------------------------------------------------
Observations: 1 2 3 6 6 6 6 6 6 1 2 1
----------------------------------------------------------------------------------------------------
Optimal Path:  Fair Fair Fair Loaded Loaded Loaded Loaded Loaded Loaded Loaded Loaded Loaded
----------------------------------------------------------------------------------------------------
